# 连接简单的 Tick 数据客户端（Tick Data Client）

> 整理自《Python for Algorithmic Trading》第 7 章 *Working with Real-Time Data and Sockets*。
>
> **阅读约定**
> - Markdown 单元格：正文 / 讲解（已译为中文）
> - Markdown 内带语法高亮的代码块：书中的**代码节选**，仅用于讲解
> - Code 单元格：**可运行的完整代码**
> - Code 单元格下方的输出区：**书中给出的运行结果**（未重新运行，仅为记录）
>
> **运行前提**：必须先在**另一个终端**里启动 tick 数据服务器（书中 "Sample Tick Data Server" 脚本），它在 `tcp://0.0.0.0:5555` 上发布数据。服务器没启动时，客户端不会报错，只会一直等待。

## 1. 概述

tick 数据服务器的代码已经相当简洁，其中最长的部分是 `InstrumentPrice` 价格模拟类。而对应的 tick 数据客户端更短，只有几行代码，完成三件事：

1. 实例化核心的 `Context` 对象；
2. 连接到发布者（publishing）套接字；
3. 订阅 `SYMBOL` 频道（channel），这也是这里唯一可用的频道。

之后在 `while` 循环中，接收字符串形式的消息并打印出来。

## 2. 初始部分：建立连接

脚本的前半部分与服务器脚本几乎是**对称**的：

```python
import zmq

context = zmq.Context()
socket = context.socket(zmq.SUB)
socket.connect('tcp://0.0.0.0:5555')
socket.setsockopt_string(zmq.SUBSCRIBE, 'SYMBOL')
```

**逐行讲解：**

1. `import zmq`：导入 ZeroMQ 库的 Python 封装。
2. `zmq.Context()`：客户端的核心对象同样是 `zmq.Context` 的实例。
3. `context.socket(zmq.SUB)`：从这里开始与服务器不同，套接字类型设为 **SUB**（订阅者）。
4. `socket.connect(...)`：连接到对应的 IP 地址与端口组合。
5. `setsockopt_string(zmq.SUBSCRIBE, 'SYMBOL')`：定义套接字订阅的**频道**。这里只有一个频道，但订阅声明仍然是必需的。在实际应用中，你可能通过一个套接字连接接收多个不同标的的数据。

## 3. 接收循环

`while` 循环的作用就是：取回服务器套接字发来的消息并打印出来。

```python
while True:
    data = socket.recv_string()
    print(data)
```

**逐行讲解：**

1. `while True`：套接字在一个无限循环中接收数据。
2. `socket.recv_string()`：核心代码行，接收数据（字符串形式的消息）。
3. `print(data)`：把数据打印到标准输出（stdout）。

## 4. 完整代码：Tick 数据客户端

下面是基于 ZeroMQ 的完整 tick 数据客户端脚本，连接到服务器脚本发布的数据。

> **注意：** 这个循环是**无限循环**，运行后不会自行结束。在 Jupyter 中需要手动点击 **Interrupt（中断内核）** 来停止。

In [1]:
#
# Python Script
# with Tick Data Client
#
# Python for Algorithmic Trading
# (c) Dr. Yves J. Hilpisch
# The Python Quants GmbH
#
import zmq

context = zmq.Context()
socket = context.socket(zmq.SUB)
socket.connect('tcp://0.0.0.0:5555')
socket.setsockopt_string(zmq.SUBSCRIBE, 'SYMBOL')

while True:
    data = socket.recv_string()
    print(data)

SYMBOL 100.00
SYMBOL 99.65
SYMBOL 99.28
SYMBOL 99.09
SYMBOL 98.76
SYMBOL 98.83
SYMBOL 98.82
SYMBOL 98.92
SYMBOL 98.57
SYMBOL 98.81
SYMBOL 98.79
SYMBOL 98.80

## 5. 结果说明

客户端脚本的输出与服务器脚本的输出**完全一致**：每条消息都是 `频道名 + 价格`（如 `SYMBOL 100.00`）。

以字符串消息形式通过套接字获取数据，只是**前提条件**，真正要基于数据完成的任务，例如**实时生成交易信号**或**数据可视化**，将在接下来的两节中介绍。

> **补充：传输其他类型的对象**
> ZeroMQ 也支持传输其他类型的对象。例如，可以通过套接字直接发送 Python 对象：默认使用 `pickle` 进行序列化和反序列化，对应的方法是 `.send_pyobj()` 和 `.recv_pyobj()`（参见 PyZMQ API 文档）。
>
> 但在实践中，各平台和数据提供商面向的是多样化的环境，Python 只是众多语言之一。因此，**基于字符串的套接字通信**更常见，通常配合 **JSON** 等标准数据格式使用。